## CHARTING A COURSE FOR MWANGAZA FILM STUDIO

### PROJECT OBJECTIVES
Help Mwangaza studio determine what kinds of movies are most successful at the box office, using available data.

### RESEARCH OBJECTIVES
Determine Market Viability – Assess which types of films resonate most with audiences and what factors contribute to commercial success.

Guide Strategic Decision-Making – Provide recommendations on which genres, styles, or storytelling approaches the new company should pursue.

To determine the optimal range of movie runtime that is most often associated with high box office success and use it to give insights on the length of an ideal movie.

### LIBRARIES

In [20]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
import numpy as np

### DATA CLEANING

In [21]:
#Loading the first dataset
df = pd.read_csv("bom.movie_gross.csv.gz")
df

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010
...,...,...,...,...,...
3382,The Quake,Magn.,6200.0,NaN,2018
3383,Edward II (2018 re-release),FM,4800.0,NaN,2018
3384,El Pacto,Sony,2500.0,NaN,2018
3385,The Swan,Synergetic,2400.0,NaN,2018


In [22]:
#looking for missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

foreign_gross     1350
domestic_gross      28
studio               5
year                 0
title                0
dtype: int64

In [23]:
df.dropna(subset=["domestic_gross","foreign_gross"],inplace=True)

In [24]:
# Looking for missing values after cleaning
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

studio            2
year              0
foreign_gross     0
domestic_gross    0
title             0
dtype: int64

In [25]:
df

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010
...,...,...,...,...,...
3275,I Still See You,LGF,1400.0,1500000,2018
3286,The Catcher Was a Spy,IFC,725000.0,229000,2018
3309,Time Freak,Grindstone,10000.0,256000,2018
3342,Reign of Judges: Title of Liberty - Concept Short,Darin Southa,93200.0,5200,2018


In [26]:
# changing the foreign gross column from strings to int and creating a new column with total gross
df["foreign_gross"] = pd.to_numeric(df["foreign_gross"], errors="coerce")
df['total_gross'] = df['domestic_gross'] + df['foreign_gross']
df

,title,studio,domestic_gross,foreign_gross,year,total_gross
0,Toy Story 3,BV,415000000.0,652000000.0,2010,1.067000e+09
1,Alice in Wonderland (2010),BV,334200000.0,691300000.0,2010,1.025500e+09
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000.0,2010,9.603000e+08
3,Inception,WB,292600000.0,535700000.0,2010,8.283000e+08
4,Shrek Forever After,P/DW,238700000.0,513900000.0,2010,7.526000e+08
...,...,...,...,...,...,...
3275,I Still See You,LGF,1400.0,1500000.0,2018,1.501400e+06
3286,The Catcher Was a Spy,IFC,725000.0,229000.0,2018,9.540000e+05
3309,Time Freak,Grindstone,10000.0,256000.0,2018,2.660000e+05
3342,Reign of Judges: Title of Liberty - Concept Short,Darin Southa,93200.0,5200.0,2018,9.840000e+04


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2009 entries, 0 to 3353
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           2009 non-null   object 
 1   studio          2007 non-null   object 
 2   domestic_gross  2009 non-null   float64
 3   foreign_gross   2004 non-null   float64
 4   year            2009 non-null   int64  
 5   total_gross     2004 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 109.9+ KB


In [28]:
df.describe()

,domestic_gross,foreign_gross,year,total_gross
count,2.009000e+03,2.004000e+03,2009.000000,2.004000e+03
mean,4.697311e+07,7.590713e+07,2013.503235,1.215769e+08
std,8.159966e+07,1.382501e+08,2.598481,2.061554e+08
min,4.000000e+02,6.000000e+02,2010.000000,4.900000e+03
25%,6.650000e+05,3.900000e+06,2011.000000,8.117750e+06
50%,1.650000e+07,1.955000e+07,2013.000000,4.210000e+07
75%,5.600000e+07,7.615000e+07,2016.000000,1.327250e+08
max,9.367000e+08,9.605000e+08,2018.000000,1.518900e+09


In [29]:
#Loading the second dataset
conn = sqlite3.connect("im.db")
cur = conn.cursor()
cur.execute("""SELECT name FROM sqlite_master WHERE type = 'table'""")
table_names = cur.fetchall()
table_names

[('movie_basics',),
 ('directors',),
 ('known_for',),
 ('movie_akas',),
 ('movie_ratings',),
 ('persons',),
 ('principals',),
 ('writers',),
 ('temp_df',)]

In [30]:
sql_df = pd.read_sql("""SELECT * FROM movie_basics;""",conn).head(15)
sql_df

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"
5,tt0111414,A Thin Life,A Thin Life,2018,75.0,Comedy
6,tt0112502,Bigfoot,Bigfoot,2017,NaN,"Horror,Thriller"
7,tt0137204,Joe Finds Grace,Joe Finds Grace,2017,83.0,"Adventure,Animation,Comedy"
8,tt0139613,O Silêncio,O Silêncio,2012,NaN,"Documentary,History"
9,tt0144449,Nema aviona za Zagreb,Nema aviona za Zagreb,2012,82.0,Biography


In [31]:
missing_null_values = sql_df.isnull().sum()
missing_null_values

movie_id           0
primary_title      0
original_title     0
start_year         0
runtime_minutes    3
genres             0
dtype: int64

In [32]:
sql_df["runtime_minutes"]= (sql_df["runtime_minutes"].median())


In [33]:
missing_null_values = sql_df.isnull().sum()
missing_null_values

movie_id           0
primary_title      0
original_title     0
start_year         0
runtime_minutes    0
genres             0
dtype: int64

In [34]:
movie_ratings = pd.read_sql("""SELECT * FROM movie_ratings;""",conn).head()
movie_ratings

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [35]:
missing_null_values = movie_ratings.isnull().sum()
missing_null_values

movie_id         0
averagerating    0
numvotes         0
dtype: int64

In [36]:
sql_df = pd.read_sql("""SELECT * FROM movie_akas ;""",conn).head()
sql_df

,movie_id,ordering,title,region,language,types,attributes,is_original_title
0,tt0369610,10,Джурасик свят,BG,bg,None,None,0.0
1,tt0369610,11,Jurashikku warudo,JP,None,imdbDisplay,None,0.0
2,tt0369610,12,Jurassic World: O Mundo dos Dinossauros,BR,None,imdbDisplay,None,0.0
3,tt0369610,13,O Mundo dos Dinossauros,BR,None,None,short title,0.0
4,tt0369610,14,Jurassic World,FR,None,imdbDisplay,None,0.0


In [37]:
query = """SELECT mr.averagerating AS avg_rating, mr.numvotes AS votes, mb.genres, mb.movie_id, mb.primary_title, mb.start_year, mb.runtime_minutes FROM movie_ratings AS mr JOIN movie_basics as mb ON mr.movie_id = mb.movie_id WHERE mr.numvotes >= 100 ORDER BY avg_rating ASC ;"""

joined = pd.read_sql(query, conn)
joined

,avg_rating,votes,genres,movie_id,primary_title,start_year,runtime_minutes
0,1.0,449,Drama,tt1611056,Hito no sabaku,2010,121.0
1,1.0,125,Comedy,tt1872215,Tunnel Rendez-vous,2011,NaN
2,1.0,520,"Fantasy,Mystery,Romance",tt3855260,Yurameku,2014,61.0
3,1.0,230,Horror,tt6010140,Desu foresuto kyofu no mori 5,2016,65.0
4,1.0,112,"Adventure,Animation,Family",tt4839424,The Autobots,2015,85.0
...,...,...,...,...,...,...,...
28748,9.6,1339,"Adventure,Biography,Documentary",tt4131686,I Want to Live,2015,106.0
28749,9.6,427,Action,tt9760512,D/O Parvathamma,2019,NaN
28750,9.7,5600,"Comedy,Drama",tt7131622,Once Upon a Time ... in Hollywood,2019,159.0
28751,9.7,639,Drama,tt8718580,Eghantham,2018,125.0


In [38]:
joined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28753 entries, 0 to 28752
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   avg_rating       28753 non-null  float64
 1   votes            28753 non-null  int64  
 2   genres           28733 non-null  object 
 3   movie_id         28753 non-null  object 
 4   primary_title    28753 non-null  object 
 5   start_year       28753 non-null  int64  
 6   runtime_minutes  27978 non-null  float64
dtypes: float64(2), int64(2), object(3)
memory usage: 1.5+ MB


In [39]:
# Merging of the dataframe(df) and joined table
merged_df = pd.merge(
    df,              # DataFrame 1: box office data
    joined,             # DataFrame 2: IMDb data
    how="inner",         # Only keep matching rows
    left_on=["title", "year"],       # from df
    right_on=["primary_title", "start_year"]  # from imdb_df
)
merged_df

,title,studio,domestic_gross,foreign_gross,year,total_gross,avg_rating,votes,genres,movie_id,primary_title,start_year,runtime_minutes
0,Toy Story 3,BV,415000000.0,652000000.0,2010,1.067000e+09,8.3,682218,"Adventure,Animation,Comedy",tt0435761,Toy Story 3,2010,103.0
1,Inception,WB,292600000.0,535700000.0,2010,8.283000e+08,8.8,1841066,"Action,Adventure,Sci-Fi",tt1375666,Inception,2010,148.0
2,Shrek Forever After,P/DW,238700000.0,513900000.0,2010,7.526000e+08,6.3,167532,"Adventure,Animation,Comedy",tt0892791,Shrek Forever After,2010,93.0
3,The Twilight Saga: Eclipse,Sum.,300500000.0,398000000.0,2010,6.985000e+08,5.0,211733,"Adventure,Drama,Fantasy",tt1325004,The Twilight Saga: Eclipse,2010,124.0
4,Iron Man 2,Par.,312400000.0,311500000.0,2010,6.239000e+08,7.0,657690,"Action,Adventure,Sci-Fi",tt1228705,Iron Man 2,2010,124.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1238,The Front Runner,Sony,2000000.0,1200000.0,2018,3.200000e+06,6.2,6372,"Biography,Drama,History",tt7074886,The Front Runner,2018,113.0
1239,Wildlife,IFC,1100000.0,2000000.0,2018,3.100000e+06,6.9,14146,Drama,tt5929754,Wildlife,2018,105.0
1240,I Still See You,LGF,1400.0,1500000.0,2018,1.501400e+06,5.7,5010,"Fantasy,Thriller",tt2160105,I Still See You,2018,98.0
1241,The Catcher Was a Spy,IFC,725000.0,229000.0,2018,9.540000e+05,6.2,4653,"Biography,Drama,War",tt4602066,The Catcher Was a Spy,2018,98.0


In [41]:
#Loading the third dataset
budget_df = pd.read_csv('tn.movie_budgets.csv.gz')
budget_df 

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"
...,...,...,...,...,...,...
5777,78,"Dec 31, 2018",Red 11,"$7,000",$0,$0
5778,79,"Apr 2, 1999",Following,"$6,000","$48,482","$240,495"
5779,80,"Jul 13, 2005",Return to the Land of Wonders,"$5,000","$1,338","$1,338"
5780,81,"Sep 29, 2015",A Plague So Pleasant,"$1,400",$0,$0


In [42]:
#checking for missing values
missing_null_values = budget_df.isnull().sum()
missing_null_values

id                   0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
dtype: int64

In [43]:
budget_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5782 entries, 0 to 5781
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 5782 non-null   int64 
 1   release_date       5782 non-null   object
 2   movie              5782 non-null   object
 3   production_budget  5782 non-null   object
 4   domestic_gross     5782 non-null   object
 5   worldwide_gross    5782 non-null   object
dtypes: int64(1), object(5)
memory usage: 271.2+ KB


In [44]:
# Convert 'release_date' to datetime format
budget_df['release_date'] = pd.to_datetime(budget_df['release_date'])

# Optional: Extract year for merging or comparison
budget_df['release_year'] = budget_df['release_date'].dt.year
# Preview cleaned data
print(budget_df.info())
print(budget_df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5782 entries, 0 to 5781
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 5782 non-null   int64         
 1   release_date       5782 non-null   datetime64[ns]
 2   movie              5782 non-null   object        
 3   production_budget  5782 non-null   object        
 4   domestic_gross     5782 non-null   object        
 5   worldwide_gross    5782 non-null   object        
 6   release_year       5782 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 316.3+ KB
None
   id release_date                                        movie  \
0   1   2009-12-18                                       Avatar   
1   2   2011-05-20  Pirates of the Caribbean: On Stranger Tides   
2   3   2019-06-07                                 Dark Phoenix   
3   4   2015-05-01                      Avengers: Age o

In [45]:
print(budget_df.head())

   id release_date                                        movie  \
0   1   2009-12-18                                       Avatar   
1   2   2011-05-20  Pirates of the Caribbean: On Stranger Tides   
2   3   2019-06-07                                 Dark Phoenix   
3   4   2015-05-01                      Avengers: Age of Ultron   
4   5   2017-12-15            Star Wars Ep. VIII: The Last Jedi   

  production_budget domestic_gross worldwide_gross  release_year  
0      $425,000,000   $760,507,625  $2,776,345,279          2009  
1      $410,600,000   $241,063,875  $1,045,663,875          2011  
2      $350,000,000    $42,762,350    $149,762,350          2019  
3      $330,600,000   $459,005,868  $1,403,013,963          2015  
4      $317,000,000   $620,181,382  $1,316,721,747          2017  


### DATA UNDERSTANDING